<a href="https://colab.research.google.com/github/Teixeiras15-collab/Atividades/blob/main/SQLALCHEMY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Atividade Prática: Desenvolvendo um Sistema de RH com SQLAlchemy

### Nível 1: Básico — Configuração e SQL Puro com Segurança

### Passo 1: Crie a conexão com um banco de dados local

In [1]:
from sqlalchemy import create_engine, text

# Crie a conexão com um banco de dados SQLite local
engine = create_engine('sqlite:///sistema_rh.db')

print("Conexão com o banco de dados 'sistema_rh.db' criada com sucesso!")

Conexão com o banco de dados 'sistema_rh.db' criada com sucesso!


### Passo 2: Abra uma transação e crie a tabela `funcionarios`

In [2]:
try:
    with engine.begin() as conn:
        # Crie a tabela 'funcionarios' com SQL puro
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS funcionarios (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                nome VARCHAR(100) NOT NULL,
                cargo VARCHAR(100) NOT NULL,
                salario DECIMAL(10, 2) NOT NULL
            )
        """))
    print("Tabela 'funcionarios' criada ou já existente.")
except Exception as e:
    print(f"Erro ao criar a tabela: {e}")

Tabela 'funcionarios' criada ou já existente.


### Passo 3 (Segurança): Simule a inserção de um novo funcionário de forma segura

In [3]:
try:
    # Dados do novo funcionário (simulando entrada de formulário web)
    novo_funcionario = {
        'nome': 'Alice Silva',
        'cargo': 'Desenvolvedora',
        'salario': 6000.00
    }

    with engine.begin() as conn:
        # Utilize um comando INSERT com parâmetros seguros
        conn.execute(
            text("INSERT INTO funcionarios (nome, cargo, salario) VALUES (:nome, :cargo, :salario)"),
            novo_funcionario
        )
    print(f"Funcionário {novo_funcionario['nome']} inserido com sucesso de forma segura.")

except Exception as e:
    print(f"Erro ao inserir funcionário: {e}")

Funcionário Alice Silva inserido com sucesso de forma segura.


### Pergunta Reflexiva (Segurança da Inserção):

**Por que nunca devemos concatenar strings diretamente no SQL (risco de injeção SQL) e como os placeholders resolvem isso?**

**Resposta:**

1.  **Risco de Injeção SQL:** Se concatenarmos strings diretamente para construir uma query SQL (ex: `f"INSERT INTO funcionarios (nome) VALUES ('{user_input}')"`), um usuário mal-intencionado pode inserir código SQL no `user_input`. Por exemplo, se `user_input` for `''); DROP TABLE funcionarios; --`, a query se tornaria `INSERT INTO funcionarios (nome) VALUES (''); DROP TABLE funcionarios; --')`, deletando a tabela.

2.  **Como Placeholders Resolvem:** Placeholders (como `:nome`, `:cargo` no SQLAlchemy ou `?` em outras bibliotecas) separam os dados dos comandos SQL. O banco de dados entende que os valores passados para os placeholders são *dados*, e não partes do comando SQL a ser executado. Isso impede que qualquer código SQL malicioso inserido nos dados seja interpretado e executado como parte da query.

### Passo 4: Valide a inserção consultando os dados com Pandas

In [4]:
import pandas as pd

try:
    # Consulte os dados e retorne como um DataFrame do Pandas
    df_funcionarios = pd.read_sql_query(text("SELECT * FROM funcionarios"), engine)
    print("Dados dos funcionários:")
    display(df_funcionarios)

except Exception as e:
    print(f"Erro ao consultar dados: {e}")

Dados dos funcionários:


,id,nome,cargo,salario
0,1,Alice Silva,Desenvolvedora,6000


## Nível 2: Intermediário — SQLAlchemy Core (Automatização Programática)

O objetivo agora é abandonar o SQL em texto e usar as estruturas Python do SQLAlchemy Core, ideais para scripts de manipulação de dados e relatórios.

### Passo 1: Defina uma nova tabela `projetos` de forma programática e crie-a no banco.

In [5]:
from sqlalchemy import Table, Column, Integer, String, MetaData, Date, ForeignKey

# Defina o objeto MetaData
metadata = MetaData()

# Defina a tabela 'projetos' programaticamente
projetos = Table(
    'projetos',
    metadata,
    Column('id', Integer, primary_key=True, autoincrement=True),
    Column('nome_projeto', String(100), nullable=False),
    Column('data_inicio', Date, nullable=False),
    Column('data_fim', Date),
    Column('funcionario_id', Integer, ForeignKey('funcionarios.id'), nullable=False)
)

# Crie a tabela 'projetos' fisicamente no banco de dados
try:
    metadata.create_all(engine)
    print("Tabela 'projetos' criada ou já existente.")
except Exception as e:
    print(f"Erro ao criar a tabela 'projetos': {e}")

Erro ao criar a tabela 'projetos': Foreign key associated with column 'projetos.funcionario_id' could not find table 'funcionarios' with which to generate a foreign key to target column 'id'


### Passo 2: Insira múltiplos projetos em lote (bulk insert).

In [6]:
from sqlalchemy import insert
import datetime

# Obter o id do funcionário 'Alice Silva' para associar aos projetos
# É importante que o funcionário já exista para que a FK seja válida
with engine.connect() as conn:
    result = conn.execute(text("SELECT id FROM funcionarios WHERE nome = 'Alice Silva'")).fetchone()
    alice_id = result[0] if result else None

if alice_id:
    # Lista de dicionários contendo múltiplos projetos
    lista_de_projetos = [
        {
            'nome_projeto': 'Sistema de Folha de Pagamento',
            'data_inicio': datetime.date(2023, 1, 15),
            'data_fim': datetime.date(2023, 6, 30),
            'funcionario_id': alice_id
        },
        {
            'nome_projeto': 'Desenvolvimento de App Mobile',
            'data_inicio': datetime.date(2023, 3, 1),
            'data_fim': None, # Projeto em andamento
            'funcionario_id': alice_id
        }
    ]

    try:
        with engine.begin() as conn:
            # Faça uma inserção em lote (bulk insert)
            conn.execute(insert(projetos), lista_de_projetos)
        print("Projetos inseridos em lote com sucesso.")

        # Verifique a inserção
        df_projetos = pd.read_sql_query(text("SELECT * FROM projetos"), engine)
        print("Dados dos projetos:")
        display(df_projetos)

    except Exception as e:
        print(f"Erro ao inserir projetos: {e}")
else:
    print("Funcionário 'Alice Silva' não encontrado, não foi possível associar os projetos.")

Erro ao inserir projetos: (sqlite3.OperationalError) no such table: projetos
[SQL: INSERT INTO projetos (nome_projeto, data_inicio, data_fim, funcionario_id) VALUES (?, ?, ?, ?)]
[parameters: [('Sistema de Folha de Pagamento', '2023-01-15', '2023-06-30', 1), ('Desenvolvimento de App Mobile', '2023-03-01', None, 1)]]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


### Passo 3: Reajuste salarial para 'Desenvolvedor Júnior'.

In [9]:
from sqlalchemy import update, DECIMAL, Table, Column, Integer, String, MetaData, Date, ForeignKey, text

# Defina a tabela 'funcionarios' programaticamente para usar o update
# Assumindo que 'funcionarios' já está definida ou pode ser redeclarada para este escopo
# Se já definida no MetaData, podemos reusar: metadata.tables['funcionarios']
funcionarios = Table(
    'funcionarios',
    metadata,
    Column('id', Integer, primary_key=True, autoincrement=True),
    Column('nome', String(100), nullable=False),
    Column('cargo', String(100), nullable=False),
    Column('salario', DECIMAL(10, 2), nullable=False)
)

# Atualize o salário dos 'Desenvolvedor Júnior'
try:
    # Primeiro, vamos adicionar um 'Desenvolvedor Júnior' para testar o reajuste
    with engine.begin() as conn:
        conn.execute(
            text("INSERT INTO funcionarios (nome, cargo, salario) VALUES (:nome, :cargo, :salario)"),
            {'nome': 'Bruno Mendes', 'cargo': 'Desenvolvedor Júnior', 'salario': 4500.00}
        )
        conn.execute(
            text("INSERT INTO funcionarios (nome, cargo, salario) VALUES (:nome, :cargo, :salario)"),
            {'nome': 'Carla Souza', 'cargo': 'Desenvolvedor Júnior', 'salario': 4800.00}
        )
    print("Adicionado 'Desenvolvedor Júnior' para demonstração de reajuste.")

    with engine.begin() as conn:
        stmt = (
            update(funcionarios)
            .where(funcionarios.c.cargo == 'Desenvolvedor Júnior')
            .values(salario=funcionarios.c.salario * 1.10) # Aumento de 10%
        )
        result = conn.execute(stmt)
        print(f"Salário de {result.rowcount} Desenvolvedor(es) Júnior reajustado(s) com sucesso.")

    # Verifique os salários atualizados
    df_funcionarios_atualizado = pd.read_sql_query(text("SELECT * FROM funcionarios"), engine)
    print("Dados dos funcionários após reajuste:")
    display(df_funcionarios_atualizado)

except Exception as e:
    print(f"Erro ao reajustar salários: {e}")

Adicionado 'Desenvolvedor Júnior' para demonstração de reajuste.
Salário de 2 Desenvolvedor(es) Júnior reajustado(s) com sucesso.
Dados dos funcionários após reajuste:


,id,nome,cargo,salario
0,1,Alice Silva,Desenvolvedora,6000
1,2,Bruno Mendes,Desenvolvedor Júnior,4950
2,3,Carla Souza,Desenvolvedor Júnior,5280


### Passo 4: Gere um relatório salarial agregando os dados (média salarial por cargo).

In [10]:
from sqlalchemy import select, func

try:
    # Construa a query para calcular a média salarial por cargo
    stmt = (
        select(funcionarios.c.cargo, func.avg(funcionarios.c.salario).label('media_salario'))
        .group_by(funcionarios.c.cargo)
    )

    with engine.connect() as conn:
        result = conn.execute(stmt)
        df_relatorio_salarial = pd.DataFrame(result.fetchall(), columns=result.keys())

    print("Relatório Salarial - Média por Cargo:")
    display(df_relatorio_salarial)

except Exception as e:
    print(f"Erro ao gerar relatório salarial: {e}")

Relatório Salarial - Média por Cargo:


,cargo,media_salario
0,Desenvolvedor Júnior,5115.0
1,Desenvolvedora,6000.0


## Nível 3: Avançado — ORM (Orientação a Objetos e Relacionamentos)

Na fase final, a turma aplicará o padrão ORM (Object-Relational Mapping), que é amplamente utilizado no desenvolvimento de aplicações modernas.

### Passo 1: Transforme as tabelas em classes Python usando ORM.

In [11]:
from sqlalchemy.orm import declarative_base, relationship, sessionmaker, Mapped, mapped_column
from sqlalchemy import Integer, String, DECIMAL, ForeignKey, Column
from typing import List

# Base declarativa para as classes ORM
Base = declarative_base()

# Definição da classe Departamento
class Departamento(Base):
    __tablename__ = 'departamentos'
    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    nome: Mapped[str] = mapped_column(String(100), unique=True, nullable=False)

    # Relação com FuncionarioORM (para navegação)
    funcionarios: Mapped[List["FuncionarioORM"]] = relationship(back_populates="departamento")

    def __repr__(self) -> str:
        return f"Departamento(id={self.id!r}, nome={self.nome!r})"

# Definição da classe FuncionarioORM
class FuncionarioORM(Base):
    __tablename__ = 'funcionarios_orm'
    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    nome: Mapped[str] = mapped_column(String(100), nullable=False)
    cargo: Mapped[str] = mapped_column(String(100), nullable=False)
    salario: Mapped[float] = mapped_column(DECIMAL(10, 2), nullable=False)

    # Chave estrangeira para a tabela de departamentos
    departamento_id: Mapped[int] = mapped_column(ForeignKey("departamentos.id"))

    # Relação com Departamento (para navegação)
    departamento: Mapped[Departamento] = relationship(back_populates="funcionarios")

    def __repr__(self) -> str:
        return f"FuncionarioORM(id={self.id!r}, nome={self.nome!r}, cargo={self.cargo!r}, salario={self.salario!r}, departamento_id={self.departamento_id!r})"

# Crie as tabelas no banco de dados
try:
    Base.metadata.create_all(engine)
    print("Tabelas 'departamentos' e 'funcionarios_orm' criadas ou já existentes via ORM.")
except Exception as e:
    print(f"Erro ao criar tabelas ORM: {e}")

Tabelas 'departamentos' e 'funcionarios_orm' criadas ou já existentes via ORM.


### Passo 2: Estabeleça a relação entre as classes.

A relação entre `Departamento` e `FuncionarioORM` foi estabelecida no Passo 1, através do `ForeignKey` em `FuncionarioORM` apontando para `departamentos.id` e o uso da função `relationship` em ambas as classes (`funcionarios` em `Departamento` e `departamento` em `FuncionarioORM`) com `back_populates`.

### Passo 3: Crie uma fábrica de sessões, instancie objetos e persista-os.

In [12]:
# Crie uma fábrica de sessões
Session = sessionmaker(bind=engine)

# Instancie uma sessão
sessao = Session()

try:
    # Crie um objeto do Departamento de TI
    dep_ti = Departamento(nome="TI")
    sessao.add(dep_ti)
    sessao.commit() # Commit para que o id do departamento seja gerado

    # Crie funcionários e adicione-os ao departamento de TI
    funcionario1 = FuncionarioORM(nome="Carlos Mendes", cargo="Engenheiro de Software", salario=7500.00, departamento=dep_ti)
    funcionario2 = FuncionarioORM(nome="Diana Faria", cargo="Analista de Dados", salario=6800.00, departamento=dep_ti)
    funcionario3 = FuncionarioORM(nome="Eduardo Lima", cargo="Desenvolvedor Júnior", salario=5200.00, departamento=dep_ti)

    sessao.add_all([funcionario1, funcionario2, funcionario3])

    # Crie um objeto do Departamento de RH e um funcionário para ele
    dep_rh = Departamento(nome="Recursos Humanos")
    sessao.add(dep_rh)
    sessao.commit()

    funcionario_rh = FuncionarioORM(nome="Fernanda Gomes", cargo="Analista de RH", salario=5500.00, departamento=dep_rh)
    sessao.add(funcionario_rh)

    # Persista todas as mudanças no banco de dados
    sessao.commit()
    print("Departamentos e funcionários inseridos com sucesso via ORM.")

except Exception as e:
    sessao.rollback() # Em caso de erro, desfaça as mudanças
    print(f"Erro ao persistir objetos ORM: {e}")
finally:
    sessao.close()

Departamentos e funcionários inseridos com sucesso via ORM.


### Passo 4: Faça uma consulta orientada a objetos para listar funcionários de "TI".

In [13]:
from sqlalchemy import select

# Reabra a sessão para a nova consulta
sessao = Session()

try:
    # Consulta para o departamento de 'TI'
    departamento_ti = sessao.execute(select(Departamento).where(Departamento.nome == "TI")).scalars().first()

    if departamento_ti:
        print(f"\nFuncionários do departamento de {departamento_ti.nome}:")
        # Acessa os funcionários relacionados através do objeto departamento
        for funcionario in departamento_ti.funcionarios:
            print(f"  - {funcionario.nome} ({funcionario.cargo}) - Salário: {funcionario.salario:.2f}")
    else:
        print("Departamento de TI não encontrado.")

    # Outra forma de consultar funcionários diretamente por cargo
    print("\nTodos os Desenvolvedores Júnior:")
    juniores = sessao.execute(select(FuncionarioORM).where(FuncionarioORM.cargo == "Desenvolvedor Júnior")).scalars().all()
    for jr in juniores:
        print(f"  - {jr.nome} (Departamento: {jr.departamento.nome})")

except Exception as e:
    print(f"Erro ao realizar consulta ORM: {e}")
finally:
    # Feche a sessão adequadamente
    sessao.close()
    print("\nSessão ORM fechada.")


Funcionários do departamento de TI:
  - Carlos Mendes (Engenheiro de Software) - Salário: 7500.00
  - Diana Faria (Analista de Dados) - Salário: 6800.00
  - Eduardo Lima (Desenvolvedor Júnior) - Salário: 5200.00

Todos os Desenvolvedores Júnior:
  - Eduardo Lima (Departamento: TI)

Sessão ORM fechada.
